In [1]:
import uproot
import awkward as ak

In [2]:
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000/BPH_MC_NANO_1.root"
file = uproot.open(file_path)
tree = file["Events"]


mom_idx = tree["BPHGenPart_genPartIdxMother"].array()
pdg_id  = tree["BPHGenPart_pdgId"].array()

In [3]:
# 1. Proteger contra índices -1 (partículas sem mãe)
# Substituímos temporariamente o -1 por 0 para evitar que acesse o final do array
safe_mom_idx = ak.where(mom_idx < 0, 0, mom_idx)

# 2. Obter o PDG ID da mãe de CADA partícula de forma vetorizada
# O array 'pdg_id' é indexado pelo array 'safe_mom_idx'
mom_pdg = pdg_id[safe_mom_idx]

In [4]:
# Para as partículas que originalmente não tinham mãe (mom_idx < 0), forçamos a mãe a ter PDG = 0
mom_pdg = ak.where(mom_idx < 0, 0, mom_pdg)

# 3. Máscara booleana: onde o PDG ID da mãe é um B0 (511 ou -511)
is_b0_daughter = abs(mom_pdg) == 511

In [5]:
# 4. Aplicar a máscara para filtrar os PDG IDs (para todos os eventos de uma vez)
b0_daughters_pdg = pdg_id[is_b0_daughter]

# Se quiser também saber as posições exatas dessas filhas no array original:
local_idx = ak.local_index(pdg_id)
b0_daughters_idx = local_idx[is_b0_daughter]

In [6]:
# Visualizando o resultado específico do evento 0
print("--- Evento 0 ---")
print(f"PDG IDs das filhas do B0: {b0_daughters_pdg[0].tolist()}")
print(f"Índices (posições) das filhas: {b0_daughters_idx[0].tolist()}")

--- Evento 0 ---
PDG IDs das filhas do B0: [-313, 22, 22]
Índices (posições) das filhas: [12, 14, 16]


In [7]:
# ---------------------------------------------------------
# BUSCA DAS NETAS DO B0 (Análise Colunar)
# ---------------------------------------------------------

# 1. Avaliamos a máscara 'is_b0_daughter' nas posições apontadas por 'safe_mom_idx'.
# Isso pergunta: "A mãe desta partícula é uma filha do B0?"
is_b0_granddaughter = is_b0_daughter[safe_mom_idx]

# 2. Proteger novamente: se a partícula não tem mãe (mom_idx < 0), ela não pode ser neta
is_b0_granddaughter = is_b0_granddaughter & (mom_idx >= 0)

# 3. Aplicar a máscara para extrair os PDGs e os índices das netas
b0_granddaughters_pdg = pdg_id[is_b0_granddaughter]
b0_granddaughters_idx = local_idx[is_b0_granddaughter]

# (Opcional) Guardar também o índice da mãe para saber de qual filha do B0 ela veio
b0_granddaughters_mom_idx = mom_idx[is_b0_granddaughter]

# Visualizando o resultado do Evento 0
print("\n--- Evento 0: Netas ---")
print(f"PDG IDs das netas do B0: {b0_granddaughters_pdg[0].tolist()}")
print(f"Índices (posições) das netas: {b0_granddaughters_idx[0].tolist()}")
print(f"Índices das mães (filhas originais): {b0_granddaughters_mom_idx[0].tolist()}")


--- Evento 0: Netas ---
PDG IDs das netas do B0: [13, -13, -321, 211]
Índices (posições) das netas: [13, 15, 24, 25]
Índices das mães (filhas originais): [14, 16, 12, 12]


In [8]:
# ---------------------------------------------------------
# FILTRANDO O EVENTO: APENAS B0, FILHAS E NETAS
# ---------------------------------------------------------

# 1. Criar uma máscara para encontrar o próprio B0
is_b0 = abs(pdg_id) == 511

# 2. Combinar as máscaras: B0 OU filha OU neta
# O operador '|' faz a operação lógica 'OR' em todos os elementos do array Awkward
keep_mask = is_b0 | is_b0_daughter | is_b0_granddaughter

# 3. Aplicar a máscara final aos arrays do evento
family_pdg = pdg_id[keep_mask]
family_original_idx = local_idx[keep_mask]
family_mom_idx = mom_idx[keep_mask]

# Visualizando o resultado do Evento 0
print("\n--- Evento 0: Família Completa do B0 ---")
print(f"PDG IDs (B0, filhas e netas): {family_pdg[0].tolist()}")
print(f"Índices originais na árvore:  {family_original_idx[0].tolist()}")
print(f"Índices originais das mães:   {family_mom_idx[0].tolist()}")


--- Evento 0: Família Completa do B0 ---
PDG IDs (B0, filhas e netas): [-511, -313, 13, 22, -13, 22, -321, 211]
Índices originais na árvore:  [0, 12, 13, 14, 15, 16, 24, 25]
Índices originais das mães:   [-1, 0, 14, 0, 16, 0, 12, 12]


# Estudo para o decaimento do J/PSI

In [17]:
import uproot
import awkward as ak

In [18]:
file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoJpsiKstar_Jpsito2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoJpsiKstar_NoFilter_2022_retry1/260427_173326/0000/BPH_MC_NANO_1.root"
file = uproot.open(file_path)
tree = file["Events"]


mom_idx = tree["BPHGenPart_genPartIdxMother"].array()
pdg_id  = tree["BPHGenPart_pdgId"].array()

In [19]:
# 1. Proteger contra índices -1 (partículas sem mãe)
# Substituímos temporariamente o -1 por 0 para evitar que acesse o final do array
safe_mom_idx = ak.where(mom_idx < 0, 0, mom_idx)

# 2. Obter o PDG ID da mãe de CADA partícula de forma vetorizada
# O array 'pdg_id' é indexado pelo array 'safe_mom_idx'
mom_pdg = pdg_id[safe_mom_idx]

In [20]:
# Para as partículas que originalmente não tinham mãe (mom_idx < 0), forçamos a mãe a ter PDG = 0
mom_pdg = ak.where(mom_idx < 0, 0, mom_pdg)

# 3. Máscara booleana: onde o PDG ID da mãe é um B0 (511 ou -511)
is_b0_daughter = abs(mom_pdg) == 511

In [21]:
# 4. Aplicar a máscara para filtrar os PDG IDs (para todos os eventos de uma vez)
b0_daughters_pdg = pdg_id[is_b0_daughter]

# Se quiser também saber as posições exatas dessas filhas no array original:
local_idx = ak.local_index(pdg_id)
b0_daughters_idx = local_idx[is_b0_daughter]

In [22]:
# Visualizando o resultado específico do evento 0
print("--- Evento 0 ---")
print(f"PDG IDs das filhas do B0: {b0_daughters_pdg[0].tolist()}")
print(f"Índices (posições) das filhas: {b0_daughters_idx[0].tolist()}")

--- Evento 0 ---
PDG IDs das filhas do B0: [443, -313]
Índices (posições) das filhas: [9, 14]


In [23]:
# ---------------------------------------------------------
# BUSCA DAS NETAS DO B0 (Análise Colunar)
# ---------------------------------------------------------

# 1. Avaliamos a máscara 'is_b0_daughter' nas posições apontadas por 'safe_mom_idx'.
# Isso pergunta: "A mãe desta partícula é uma filha do B0?"
is_b0_granddaughter = is_b0_daughter[safe_mom_idx]

# 2. Proteger novamente: se a partícula não tem mãe (mom_idx < 0), ela não pode ser neta
is_b0_granddaughter = is_b0_granddaughter & (mom_idx >= 0)

# 3. Aplicar a máscara para extrair os PDGs e os índices das netas
b0_granddaughters_pdg = pdg_id[is_b0_granddaughter]
b0_granddaughters_idx = local_idx[is_b0_granddaughter]

# (Opcional) Guardar também o índice da mãe para saber de qual filha do B0 ela veio
b0_granddaughters_mom_idx = mom_idx[is_b0_granddaughter]

# Visualizando o resultado do Evento 0
print("\n--- Evento 0: Netas ---")
print(f"PDG IDs das netas do B0: {b0_granddaughters_pdg[0].tolist()}")
print(f"Índices (posições) das netas: {b0_granddaughters_idx[0].tolist()}")
print(f"Índices das mães (filhas originais): {b0_granddaughters_mom_idx[0].tolist()}")


--- Evento 0: Netas ---
PDG IDs das netas do B0: [22, 22, -321, 211]
Índices (posições) das netas: [11, 13, 17, 18]
Índices das mães (filhas originais): [9, 9, 14, 14]


In [25]:
# ---------------------------------------------------------
# BUSCA DAS BISNETAS DO B0 (Análise Colunar)
# ---------------------------------------------------------

# 1. Avaliamos a máscara 'is_b0_granddaughter' (que criamos no passo anterior) 
# nas posições apontadas por 'safe_mom_idx'.
# Isso pergunta: "A mãe desta partícula é uma neta do B0?"
is_b0_greatgranddaughter = is_b0_granddaughter[safe_mom_idx]

# 2. Proteger novamente: se a partícula não tem mãe (mom_idx < 0), ela não pode ser bisneta
is_b0_greatgranddaughter = is_b0_greatgranddaughter & (mom_idx >= 0)

# 3. Aplicar a máscara para extrair os PDGs e os índices das bisnetas
b0_greatgranddaughters_pdg = pdg_id[is_b0_greatgranddaughter]
b0_greatgranddaughters_idx = local_idx[is_b0_greatgranddaughter]

# (Opcional) Guardar também o índice da mãe para saber de qual neta do B0 ela veio
b0_greatgranddaughters_mom_idx = mom_idx[is_b0_greatgranddaughter]

# Visualizando o resultado do Evento 0
print("\n--- Evento 0: Bisnetas ---")
print(f"PDG IDs das bisnetas do B0: {b0_greatgranddaughters_pdg[0].tolist()}")
print(f"Índices (posições) das bisnetas: {b0_greatgranddaughters_idx[0].tolist()}")
print(f"Índices das mães (netas originais): {b0_greatgranddaughters_mom_idx[0].tolist()}")


--- Evento 0: Bisnetas ---
PDG IDs das bisnetas do B0: [-13, 13]
Índices (posições) das bisnetas: [10, 12]
Índices das mães (netas originais): [11, 13]


In [26]:
# ---------------------------------------------------------
# FILTRANDO O EVENTO: APENAS B0, FILHAS, NETAS E BISNETAS
# ---------------------------------------------------------

# 1. Criar uma máscara para encontrar o próprio B0
is_b0 = abs(pdg_id) == 511

# 2. Combinar as máscaras: B0 OU filha OU neta OU bisneta
# O operador '|' faz a operação lógica 'OR' em todos os elementos do array Awkward
keep_mask = is_b0 | is_b0_daughter | is_b0_granddaughter | is_b0_greatgranddaughter

# 3. Aplicar a máscara final aos arrays do evento
family_pdg = pdg_id[keep_mask]
family_original_idx = local_idx[keep_mask]
family_mom_idx = mom_idx[keep_mask]

# Visualizando o resultado do Evento 0
print("\n--- Evento 0: Família Completa do B0 (até bisnetas) ---")
print(f"PDG IDs (B0, filhas, netas e bisnetas): {family_pdg[0].tolist()}")
print(f"Índices originais na árvore:  {family_original_idx[0].tolist()}")
print(f"Índices originais das mães:   {family_mom_idx[0].tolist()}")


--- Evento 0: Família Completa do B0 (até bisnetas) ---
PDG IDs (B0, filhas, netas e bisnetas): [-511, 443, -13, 22, 13, 22, -313, -321, 211]
Índices originais na árvore:  [1, 9, 10, 11, 12, 13, 14, 17, 18]
Índices originais das mães:   [-1, 1, 11, 9, 13, 9, 1, 14, 14]
